# 黒い背景・白い立体表面で3D表示

保存済みの `volume_data.npz` を使います。Bessel生成・圧縮・伝搬の再計算は行いません。

目標は**幾何形状**、直接法と提案法は**同じ強度閾値を超えた領域の表面**として表示します。強度の数値評価は元のデータで行い、表示の二値化・間引きは評価に戻しません。厚みが見えるよう座標軸の表示比率を変えますが、物理座標と単位は保持します。


In [ ]:
import importlib
from pathlib import Path
from IPython.display import Image, display
import lsh_multisheet_3d as viewer
viewer = importlib.reload(viewer)

# 最も新しい保存済み3Dデータを選ぶ。特定の結果を見たい場合は下の行で指定する。
files = list(Path('.').glob('lsh_multisheet_outputs*/volume_data.npz'))
if not files:
    raise FileNotFoundError('先にlsh_multisheet.ipynbを実行し、volume_data.npzを保存してください。')
RESULT_DIR = max(files, key=lambda path: path.stat().st_mtime).parent
# RESULT_DIR = Path('lsh_multisheet_outputs_rank450_20261005_160733_582182')

KIND = 'cgh'  # CGH再生強度。CGHを介さない場を表示する場合は'complex'。
THRESHOLD = 0.20  # 直接法の最大強度の20%。両手法に同じ値を使う。
ELEVATION = 25.0
AZIMUTH = -65.0
MAX_DISPLAY_SHAPE = (128, 96, 160)  # [z,y,x]の表示用上限。増やすと細かいが重くなる。
BOX_ASPECT = (1.5, 1.0, 0.9)  # 表示上のz/x/y軸の長さ比。実際の物理寸法比ではない。
print('表示する結果:', RESULT_DIR.resolve())


In [ ]:
paths = viewer.export_from_saved(
    RESULT_DIR, kind=KIND, threshold_fraction=THRESHOLD,
    elevation=ELEVATION, azimuth=AZIMUTH,
    max_shape=MAX_DISPLAY_SHAPE, box_aspect=BOX_ASPECT,
)
print('表示条件:', Path(paths['settings']).resolve())
display(Image(filename=paths['comparison'], width=1500))


In [ ]:
# 各図を個別に保存するので、ゼミのスライドへそのまま貼り付けられる。
for name in ('target', 'direct', 'proposed'):
    print(name, '→', Path(paths[name]).resolve())
    display(Image(filename=paths[name], width=700))


## 回転したい場合

ノートブックの `ELEVATION` / `AZIMUTH` を変更し、表示セルだけを再実行します。MatplotlibのGUIでマウス回転したい場合は、ターミナルから次のように起動できます。

```powershell
python lsh_multisheet_3d.py lsh_multisheet_outputs_rank450_20261005_160733_582182 --show
```

出力は結果フォルダの `3d_surface/` 内です。白い面は閾値処理による表示で、光強度が一様になったことを意味しません。定量評価には元の `summary.json` と断面画像を使ってください。
